# Induction Heads in GPT-2 Small

Companion notebook for the [Mechanistic Interpretability](https://codemia.io/courses/applied_ai_systems/mechanistic_interpretability) lesson on Codemia.

**What you will build**: a walkthrough of how to find an induction head in GPT-2 small using TransformerLens. You will visualize attention on a repeated sequence, compute an induction score for every head, and verify your finding with an activation patching ablation.

**Why induction heads matter**: they are the simplest learned algorithm inside a transformer that we fully understand. They implement in-context learning at its most primitive level — "I saw `A B` earlier, so after seeing `A` again I should predict `B`". Finding them from activations alone is the canonical mechanistic interpretability exercise.

**References**:
- Olsson et al., [In-context Learning and Induction Heads](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html) (Anthropic, 2022)
- Elhage et al., [A Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html) (Anthropic, 2021)
- [TransformerLens docs](https://transformerlensorg.github.io/TransformerLens/)

**Time**: ~60 minutes  **GPU required**: yes (T4 is fine — GPT-2 small is 124M params)

## Setup

In [ ]:
%pip install -q transformer_lens torch matplotlib

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer

torch.set_grad_enabled(False)  # pure inference — interpretability doesn't need gradients
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
# HookedTransformer is TransformerLens's wrapper around HuggingFace models that
# exposes every intermediate activation via a hook system. "gpt2" is GPT-2 small
# (12 layers, 12 heads, 768 dim = 124M params).
model = HookedTransformer.from_pretrained('gpt2', device=device)
model.eval()
print(f'layers={model.cfg.n_layers}, heads={model.cfg.n_heads}, d_model={model.cfg.d_model}')

## Concept recap

An **induction head** is a specific attention pattern that implements the algorithm:

> If the current token is `A`, look back in the context for previous occurrences of `A`, and attend to the token that *followed* it (`B`). Copy information from that `B` position so the model can predict `B` as the next token.

This two-step "match then copy" needs a *pair* of heads in adjacent layers (a "previous token head" in an earlier layer plus the induction head itself), which is why induction heads only appear in 2+ layer models. In GPT-2 small, the classic induction heads live in layers 5 and 6.

To detect them, we use a diagnostic: feed the model a sequence `[random tokens] ++ [random tokens]` — exactly the same random prefix repeated twice. An induction head will show a distinctive diagonal attention pattern shifted by the period length, because every token in the second half finds its match in the first half.

## Implementation

Step 1: generate a repeated sequence of random tokens and run the model with caching to capture all attention patterns.

In [ ]:
def make_repeated_tokens(seq_len=50, batch=1):
    # Sample `seq_len` random token ids from the model vocab, then duplicate them.
    # Adding the BOS token at position 0 gives the model a valid starting context.
    vocab = model.cfg.d_vocab
    half = torch.randint(0, vocab, (batch, seq_len), device=device)
    bos = torch.full((batch, 1), model.tokenizer.bos_token_id, device=device, dtype=half.dtype)
    return torch.cat([bos, half, half], dim=-1)  # [batch, 2*seq_len + 1]

tokens = make_repeated_tokens(seq_len=50)
print('shape:', tokens.shape)
print('first 10 tokens:', tokens[0, :10].tolist())

In [ ]:
# run_with_cache: forward pass that stashes every intermediate activation
# (attention scores, residuals, MLP outputs, etc.) in a dict-like ActivationCache.
logits, cache = model.run_with_cache(tokens)
print('logits:', tuple(logits.shape))
print('cache keys sample:', list(cache.keys())[:6])

### Loss on the repeated half

A working induction head should make the second-half tokens much easier to predict than the first-half tokens. Compare the cross-entropy loss on both halves to confirm the model is doing in-context learning at all.

In [ ]:
def per_token_loss(logits, tokens):
    # Standard next-token loss: compare logits[t] against tokens[t+1].
    logprobs = logits.log_softmax(dim=-1)
    # Gather the log-prob of each actual next token.
    # Shape juggling: predict token at t+1 from position t.
    preds = logprobs[:, :-1]                                # [B, T-1, V]
    targets = tokens[:, 1:]                                 # [B, T-1]
    return -preds.gather(-1, targets.unsqueeze(-1)).squeeze(-1)

losses = per_token_loss(logits, tokens)[0]  # squeeze batch dim
# Split into first-half predictions vs second-half predictions.
# The 51 tokens of the first half have positions 0-50, predicting positions 1-51.
half = (tokens.shape[-1] - 1) // 2
print(f'mean loss first half:  {losses[:half].mean().item():.3f}')
print(f'mean loss second half: {losses[half:].mean().item():.3f}')

Second-half loss should be much lower (~0.5-2) than first-half (~9-10). The model can "remember" the repeated pattern — that is in-context learning in action, courtesy of induction heads.

### Attention pattern on a single head

Let's peek at layer 5, head 5 (a known induction head in GPT-2 small). An induction head's attention map has a diagonal line offset by `seq_len` — every position in the second half attends back to its corresponding position in the first half, plus one (the token *after* the match).

In [ ]:
def attn_of(layer, head):
    # cache['pattern', layer] has shape [batch, head, query_pos, key_pos]
    return cache['pattern', layer][0, head].cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (l, h) in zip(axes, [(5, 1), (5, 5), (6, 9)]):
    im = ax.imshow(attn_of(l, h), aspect='auto', cmap='viridis')
    ax.set_title(f'layer {l}, head {h}')
    ax.set_xlabel('key pos')
    ax.set_ylabel('query pos')
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

### Induction score for every head

Define an induction score for head `(l, h)`: given a sequence of period `T`, look at the attention value at offset `T-1` on the diagonal (each query position `q` in the second half attending to position `q - T + 1`). Average across the second-half positions. A perfect induction head gives a score near 1.0. Most heads give a score near 0.

In [ ]:
def induction_score(pattern, seq_len):
    # pattern: [query_pos, key_pos] for a single head
    # The diagonal we want is the one where key_pos = query_pos - seq_len + 1.
    # That's the position of the token immediately AFTER the original occurrence.
    diagonal = pattern.diagonal(offset=-(seq_len - 1))
    # Only take second-half queries (they are the only ones that have a valid match).
    return float(diagonal[seq_len:].mean())

seq_len = 50
scores = np.zeros((model.cfg.n_layers, model.cfg.n_heads))
for l in range(model.cfg.n_layers):
    for h in range(model.cfg.n_heads):
        scores[l, h] = induction_score(attn_of(l, h), seq_len)

plt.figure(figsize=(8, 5))
plt.imshow(scores, cmap='plasma', aspect='auto')
plt.colorbar(label='induction score')
plt.xlabel('head')
plt.ylabel('layer')
plt.title('Induction scores across all heads')
plt.show()

top5 = np.argsort(scores.flatten())[-5:][::-1]
print('top 5 induction heads (layer, head, score):')
for idx in top5:
    l, h = divmod(int(idx), model.cfg.n_heads)
    print(f'  L{l}.H{h}  score={scores[l, h]:.3f}')

You should see one or two heads — typically in layers 5 and 6 — with scores well above 0.3 while the rest are near 0. Those are induction heads.

## Causal verification with activation patching

Correlation is not causation. A high induction score *could* be coincidence. The decisive test is: if we zero-ablate an induction head, does the model's performance on the repeated-sequence task actually get worse?

We use TransformerLens's hook system to clamp the attention output of a single head to zero, then re-measure the second-half loss.

In [ ]:
def ablate_head(layer, head, tokens):
    # A hook function receives the activation tensor and the HookPoint object,
    # and must return a modified tensor.
    # cache['z', layer] has shape [batch, pos, head, d_head]
    # Setting the `head` index to zero effectively removes that head's contribution
    # to the residual stream after the output projection.
    def hook(z, hook):
        z[:, :, head, :] = 0
        return z
    # run_with_hooks attaches the hook for one forward pass then removes it.
    return model.run_with_hooks(
        tokens,
        fwd_hooks=[(f'blocks.{layer}.attn.hook_z', hook)],
    )

# Baseline loss on the second half (no ablation).
baseline_logits = model(tokens)
baseline_loss = per_token_loss(baseline_logits, tokens)[0, half:].mean().item()
print(f'baseline second-half loss: {baseline_loss:.3f}')

In [ ]:
# Measure the loss change from ablating each of the top induction heads
# (and a random non-induction head for contrast).
top_heads = [tuple(divmod(int(idx), model.cfg.n_heads)) for idx in top5[:2]]
control_head = (0, 0)  # very unlikely to be an induction head
heads_to_test = top_heads + [control_head]

for (l, h) in heads_to_test:
    ablated = ablate_head(l, h, tokens)
    loss_after = per_token_loss(ablated, tokens)[0, half:].mean().item()
    delta = loss_after - baseline_loss
    label = 'INDUCTION' if (l, h) in top_heads else 'CONTROL'
    print(f'{label}  L{l}.H{h}  loss: {baseline_loss:.3f} -> {loss_after:.3f}  ({delta:+.3f})')

If the heads we identified really are induction heads, ablating them should *significantly* raise the second-half loss (delta > 0.5 is a strong effect on a tiny model like GPT-2). The control head should show near-zero delta. That's the causal evidence.

## Variations

In [ ]:
# Variation 1: longer repeated sequences. Induction heads should stay sharp.
for L in [20, 50, 100]:
    toks = make_repeated_tokens(seq_len=L)
    _, c = model.run_with_cache(toks)
    pat = c['pattern', 5][0, 5].cpu().numpy()
    print(f'seq_len={L:3d}  L5.H5 score={induction_score(pat, L):.3f}')

In [ ]:
# Variation 2: non-repeated random tokens. Induction heads should go SILENT
# because there is no pattern to match.
random_toks = torch.randint(0, model.cfg.d_vocab, (1, 101), device=device)
_, rc = model.run_with_cache(random_toks)
rand_pat = rc['pattern', 5][0, 5].cpu().numpy()
# Score the same diagonal shape for comparison.
print(f'repeated L5.H5:    {induction_score(attn_of(5, 5), 50):.3f}')
print(f'non-repeated L5.H5: {induction_score(rand_pat, 50):.3f}')

## What to try next

- Find the **previous-token head** that feeds the induction head. It lives in an earlier layer (typically layer 4 in GPT-2 small) and its attention map is a diagonal shifted by 1. The induction head reads from wherever the previous-token head wrote.
- Repeat this analysis on larger models (`gpt2-medium`, `gpt2-xl`, `pythia-410m`). Induction heads appear at consistent fractional depth across model sizes — this is one of the Anthropic team's surprise findings.
- Try **path patching** instead of ablation. Instead of zeroing the head, replace its activation on a clean input with the activation from a corrupted input. This isolates which *information flow* matters, not just whether the head fires.
- Read the Anthropic paper "In-context Learning and Induction Heads" and reproduce Figure 1: the moment in training when in-context loss drops sharply coincides with induction heads forming. You'd need a trained checkpoint stream like Pythia for this.